# Pipeline NER ProcteMist — PlanTL-GOB-ES/bsc-bio-ehr-es

Este cuaderno resume un flujo completo de trabajo para reconocimiento de entidades en textos clinicos.
Reune pasos de configuracion, entrenamiento, inferencia y evaluacion en un solo lugar.

## Contenido

1. [Entorno y dependencias](#1-entorno-y-dependencias)
2. [Configuracion](#2-configuracion)
   - [Rutas y dataset](#21-rutas-y-dataset)
   - [Etiquetas y carga de datos](#22-etiquetas-y-carga-de-datos)
   - [Segmentacion y alineacion de etiquetas](#23-segmentacion-y-alineacion-de-etiquetas)
   - [Hiperparametros y tokenizador](#24-hiperparametros-y-tokenizador)
3. [Entrenamiento](#3-entrenamiento)
   - [Metricas de evaluacion](#31-metricas-de-evaluacion)
   - [Discriminative fine-tuning](#32-discriminative-fine-tuning)
   - [Loop k-fold multi-semilla](#33-loop-k-fold-multi-semilla)
   - [Resumen del ensamble](#34-resumen-del-ensamble)
4. [Inferencia](#4-inferencia)
   - [Funcion de inferencia por oraciones](#41-funcion-de-inferencia-por-oraciones)
   - [Ejecucion del ensamble](#42-ejecucion-del-ensamble)
5. [Evaluacion](#5-evaluacion)
   - [Evaluacion estricta por offsets](#51-evaluacion-estricta-por-offsets)
   - [Evaluacion por solapamiento (IoU)](#52-evaluacion-por-solapamiento-iou)

## 1. Entorno y dependencias

Instalacion de paquetes necesarios e importacion de librerias.

In [1]:
%pip install -q evaluate seqeval spacy datasets transformers
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 42.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import evaluate
import numpy as np
import pandas as pd
import spacy
import torch
from torch.optim import AdamW
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU disponible: True
GPU: Tesla T4


## 2. Configuracion

### 2.1. Rutas y dataset

Definicion de rutas al dataset ProcteMist y seleccion del modelo base.

In [ ]:
# Configuracion global de rutas
PROJECT_ROOT = "/kaggle/input/datasets/user"
PROCTEMIST_ROOT = f"{PROJECT_ROOT}/proctemist"

DATA_PATHS = {
    "train_jsonl": f"{PROCTEMIST_ROOT}/proctemist_train.jsonl",
    "test_jsonl": f"{PROCTEMIST_ROOT}/proctemist_test.jsonl",
    "text_files_train_dir": f"{PROCTEMIST_ROOT}/text_files_train",
    "text_files_test_dir": f"{PROCTEMIST_ROOT}/text_files_test",
    "gs_mentions_tsv": f"{PROCTEMIST_ROOT}/medprocner_tsv_test_subtask1.tsv",
}

# Configuración del modelo base
BASE_MODEL = "PlanTL-GOB-ES/bsc-bio-ehr-es"

print("Rutas configuradas:")
for k, v in DATA_PATHS.items():
    print(f"  - {k}: {v}")
print(f"Modelo base: {BASE_MODEL}")

### 2.2. Etiquetas y carga de datos

Mapeo BIO de etiquetas y carga del JSONL de entrenamiento.

In [4]:
id2label = {0: "B-PROCEDIMIENTO", 1: "I-PROCEDIMIENTO", 2: "O"}
label2id = {"B-PROCEDIMIENTO": 0, "I-PROCEDIMIENTO": 1, "O": 2}
label_list = [id2label[i] for i in range(len(id2label))]

nlp_spacy = spacy.load("es_core_news_md")

from datasets import load_dataset as _load_dataset
train_full = _load_dataset("json", data_files=DATA_PATHS["train_jsonl"], split="train")

print(f"Etiquetas: {label2id}")
print(f"Documentos de entrenamiento: {len(train_full)}")

Generating train split: 0 examples [00:00, ? examples/s]

Etiquetas: {'B-PROCEDIMIENTO': 0, 'I-PROCEDIMIENTO': 1, 'O': 2}
Documentos de entrenamiento: 749


### 2.3. Segmentacion y alineacion de etiquetas

Funciones de segmentacion por oraciones con spaCy y alineacion de etiquetas BIO durante la tokenizacion.

In [5]:
def split_by_sentences(text, tokens, labels, nlp_spacy):
    """Divide tokens y etiquetas de un documento en segmentos de oracion usando spaCy."""
    doc = nlp_spacy(text)
    sentences = list(doc.sents)

    if len(sentences) <= 1:
        return [(tokens, labels)]

    token_char_starts = []
    search_pos = 0
    for tok in tokens:
        idx = text.find(tok, search_pos)
        if idx == -1:
            return [(tokens, labels)]
        token_char_starts.append(idx)
        search_pos = idx + len(tok)

    results = []
    for sent in sentences:
        sent_start = sent.start_char
        sent_end = sent.end_char
        sent_token_indices = [
            i for i, cs in enumerate(token_char_starts)
            if sent_start <= cs < sent_end
        ]
        if not sent_token_indices:
            continue
        sent_tokens = [tokens[i] for i in sent_token_indices]
        sent_labels = [labels[i] for i in sent_token_indices]
        results.append((sent_tokens, sent_labels))

    return results if results else [(tokens, labels)]


def tokenize_and_align_labels(examples, tok, nlp_spacy, max_length=512):
    """Tokeniza por oraciones con truncation=True y propaga B->I en subtokens."""
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for doc_idx in range(len(examples["tokens"])):
        text = examples["text"][doc_idx]
        tokens = examples["tokens"][doc_idx]
        ner_tags = examples["ner_tags"][doc_idx]

        sent_chunks = split_by_sentences(text, tokens, ner_tags, nlp_spacy)

        for sent_tokens, sent_labels in sent_chunks:
            tokenized = tok(
                [sent_tokens],
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                padding=False,
            )

            word_ids = tokenized.word_ids(batch_index=0)
            previous_word_idx = None
            label_ids = []

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(sent_labels[word_idx])
                else:
                    prev_label = sent_labels[word_idx]
                    label_ids.append(1 if prev_label == 0 else prev_label)
                previous_word_idx = word_idx

            all_input_ids.append(tokenized["input_ids"][0])
            all_attention_masks.append(tokenized["attention_mask"][0])
            all_labels.append(label_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }

### 2.4. Hiperparametros y tokenizador

Configuracion del experimento: hiperparametros de entrenamiento y carga del tokenizador.

In [6]:
BASE_MODEL_TAG = BASE_MODEL.split("/")[-1]

MAX_EPOCHS          = 20
BATCH_SIZE          = 16
LEARNING_RATE       = 8.516e-5
LR_LAYER_DECAY      = 0.95
LR_ENCODER_GROUPS   = 3
DROPOUT             = 0.1
WEIGHT_DECAY        = 0.1844
WARMUP_RATIO        = 0.1
EARLY_STOPPING_PATIENCE   = 5
EARLY_STOPPING_THRESHOLD  = 1e-4

K_FOLDS             = 5
CV_SPLIT_SEED       = 42
SEEDS               = [123, 4242]
ENSEMBLE_VOTING_RATIO = 0.5

RESULTS_DIR        = f"results_{BASE_MODEL_TAG}_kfold_multiseed"
MODEL_OUTPUT_PREFIX = f"{BASE_MODEL_TAG}-proctemist-ner"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

config = AutoConfig.from_pretrained(
    BASE_MODEL,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
    classifier_dropout=DROPOUT,
    attn_implementation="sdpa",
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    add_prefix_space=True,
    do_lower_case=False,
    keep_accents=True,
    model_max_length=config.max_position_embeddings,
)

hyperparams = {
    "base_model": BASE_MODEL, "base_model_tag": BASE_MODEL_TAG,
    "max_epochs": MAX_EPOCHS, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "lr_layer_decay": LR_LAYER_DECAY,
    "lr_encoder_groups": LR_ENCODER_GROUPS, "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "k_folds": K_FOLDS, "cv_split_seed": CV_SPLIT_SEED,
    "seeds": SEEDS, "ensemble_voting_ratio": ENSEMBLE_VOTING_RATIO,
}
with open(f"{RESULTS_DIR}/hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(hyperparams, f, ensure_ascii=False, indent=2)

print(f"Modelo: {BASE_MODEL} | Max pos embeddings: {config.max_position_embeddings}")
print(f"Resultados en: {RESULTS_DIR}")

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Modelo: PlanTL-GOB-ES/bsc-bio-ehr-es | Max pos embeddings: 514
Resultados en: results_bsc-bio-ehr-es_kfold_multiseed


## 3. Entrenamiento

### 3.1. Metricas de evaluacion

Definicion de la metrica seqeval para evaluacion NER durante el entrenamiento.

In [7]:
metric_fn = evaluate.load("seqeval", trust_remote_code=True)


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[pred] for (pred, la) in zip(prediction, label) if la != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[la] for (_, la) in zip(prediction, label) if la != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric_fn.compute(
        predictions=true_predictions,
        references=true_labels,
        zero_division=0.0,
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### 3.2. Discriminative fine-tuning

Asignacion de tasas de aprendizaje diferenciadas por profundidad de capa.

In [8]:
def create_discriminative_optimizer(model):
    named_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    layer_re = re.compile(r"\.(?:encoder\.layer|layer|layers|block|h)\.(\d+)\.")
    layer_ids = [int(m.group(1)) for n, _ in named_params for m in [layer_re.search(n.lower())] if m]
    max_layer_id = max(layer_ids) if layer_ids else 0

    groups = {}
    for name, param in named_params:
        lname = name.lower()
        if "classifier" in lname or "crf" in lname:
            bucket, lr = "head", LEARNING_RATE
        elif "embed" in lname:
            bucket, lr = "embeddings", LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS + 1))
        else:
            match = layer_re.search(lname)
            if match and max_layer_id > 0:
                zone = min(int((int(match.group(1)) / max_layer_id) * LR_ENCODER_GROUPS), LR_ENCODER_GROUPS - 1)
                bucket, lr = f"encoder_{zone}", LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS - zone))
            else:
                bucket, lr = "head", LEARNING_RATE

        if bucket not in groups:
            groups[bucket] = {"params": [], "lr": float(lr), "param_count": 0, "tensor_count": 0}
        groups[bucket]["params"].append(param)
        groups[bucket]["param_count"] += param.numel()
        groups[bucket]["tensor_count"] += 1

    optimizer = AdamW(
        [{"params": g["params"], "lr": g["lr"], "weight_decay": WEIGHT_DECAY} for g in groups.values()],
        lr=LEARNING_RATE,
        fused=torch.cuda.is_available(),
    )
    order = ["embeddings"] + [f"encoder_{i}" for i in range(LR_ENCODER_GROUPS)] + ["head"]
    summary = [
        {"bucket": b, "lr": groups[b]["lr"], "param_count": groups[b]["param_count"], "tensor_count": groups[b]["tensor_count"]}
        for b in order if b in groups
    ]
    return optimizer, summary

### 3.3. Loop k-fold multi-semilla

Entrenamiento por folds y semillas con early stopping. Los modelos resultantes forman el ensamble.

In [9]:
def make_kfold_indices(n_samples, k_folds, split_seed):
    rng = np.random.default_rng(split_seed)
    indices = np.arange(n_samples)
    rng.shuffle(indices)
    fold_sizes = np.full(k_folds, n_samples // k_folds, dtype=int)
    fold_sizes[: n_samples % k_folds] += 1
    folds, current = [], 0
    for size in fold_sizes:
        val_idx = indices[current: current + size]
        train_idx = np.concatenate((indices[:current], indices[current + size:]))
        folds.append((train_idx, val_idx))
        current += size
    return folds


fold_seed_results = []
ensemble_models = []
folds = make_kfold_indices(len(train_full), K_FOLDS, CV_SPLIT_SEED)

print(f"Entrenamiento k-fold multi-semilla | docs={len(train_full)} | folds={K_FOLDS} | seeds={SEEDS}")

for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):
    train_raw = train_full.select(train_idx.tolist())
    val_raw   = train_full.select(val_idx.tolist())

    map_kwargs = dict(batched=True, remove_columns=train_full.column_names)
    fn = lambda x: tokenize_and_align_labels(x, tokenizer, nlp_spacy, max_length=512)
    train_ds = train_raw.map(fn, **map_kwargs)
    val_ds   = val_raw.map(fn, **map_kwargs)

    print(f"\nFold {fold_idx}/{K_FOLDS} | train={len(train_raw)} docs / {len(train_ds)} seqs | val={len(val_raw)} docs / {len(val_ds)} seqs")

    for seed in SEEDS:
        print(f"  Seed {seed}...")
        set_seed(seed)
        start_time = time.time()

        model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, config=config)
        model.gradient_checkpointing_enable()
        optimizer, lr_summary = create_discriminative_optimizer(model)

        output_dir = f"{RESULTS_DIR}/{MODEL_OUTPUT_PREFIX}-fold{fold_idx}-seed{seed}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            logging_strategy="epoch",
            save_strategy="epoch",
            num_train_epochs=MAX_EPOCHS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            save_total_limit=1,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_SIZE,
            dataloader_num_workers=2,
            dataloader_prefetch_factor=4,
            dataloader_persistent_workers=True,
            seed=seed,
            bf16=True,
            save_only_model=True,
            report_to="none",
        )

        trainer = Trainer(
            model, training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            data_collator=DataCollatorForTokenClassification(tokenizer),
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            )],
            optimizers=(optimizer, None),
        )
        trainer.train()

        model_dir = trainer.state.best_model_checkpoint or output_dir
        val_metrics = trainer.evaluate(val_ds)
        best_logs = [l for l in trainer.state.log_history if "eval_f1" in l]
        best_f1   = max((l["eval_f1"] for l in best_logs), default=float("nan"))
        elapsed   = (time.time() - start_time) / 60

        row = {
            "fold": fold_idx, "seed": seed,
            "train_docs": len(train_raw), "val_docs": len(val_raw),
            "train_sequences": len(train_ds), "val_sequences": len(val_ds),
            "best_eval_f1": best_f1,
            "eval_precision": val_metrics.get("eval_precision", float("nan")),
            "eval_recall":    val_metrics.get("eval_recall",    float("nan")),
            "eval_f1":        val_metrics.get("eval_f1",        float("nan")),
            "eval_accuracy":  val_metrics.get("eval_accuracy",  float("nan")),
            "eval_loss":      val_metrics.get("eval_loss",      float("nan")),
            "elapsed_min": elapsed, "model_dir": model_dir,
        }
        fold_seed_results.append(row)
        ensemble_models.append({"fold": fold_idx, "seed": seed, "model_dir": model_dir, "eval_f1": row["eval_f1"]})

        print(f"    fold={fold_idx} seed={seed} | best_f1={best_f1:.4f} | eval_f1={row['eval_f1']:.4f} | {elapsed:.1f} min")

        del trainer, model, optimizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if not fold_seed_results:
    raise RuntimeError("No se entreno ningun modelo fold-semilla.")

df_ensemble_results = (
    pd.DataFrame(fold_seed_results)
    .sort_values(["eval_f1", "fold", "seed"], ascending=[False, True, True])
    .reset_index(drop=True)
)
df_ensemble_results.to_csv(f"{RESULTS_DIR}/ensemble_fold_seed_summary.csv", index=False)
with open(f"{RESULTS_DIR}/ensemble_fold_seed_summary.json", "w", encoding="utf-8") as f:
    json.dump(fold_seed_results, f, ensure_ascii=False, indent=2)

print(f"\nModelos en ensamble: {len(ensemble_models)}")
print(df_ensemble_results[["fold","seed","eval_f1","best_eval_f1","elapsed_min"]].to_string(index=False))

Entrenamiento k-fold multi-semilla | docs=749 | folds=5 | seeds=[123, 4242]


Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


Fold 1/5 | train=599 docs / 9419 seqs | val=150 docs / 2292 seqs
  Seed 123...


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

    fold=5 seed=4242 | best_f1=0.7696 | eval_f1=0.7696 | 54.1 min

Modelos en ensamble: 10
 fold  seed  eval_f1  best_eval_f1  elapsed_min
    2   123 0.781192      0.781192    53.803233
    5   123 0.780546      0.780546    53.987058
    5  4242 0.769592      0.769592    54.072973
    1   123 0.765818      0.765818    54.098377
    1  4242 0.762105      0.762217    51.440399
    4  4242 0.761734      0.761905    54.273129
    2  4242 0.761683      0.762436    53.740306
    4   123 0.754077      0.754077    46.092830
    3   123 0.740099      0.740206    53.536999
    3  4242 0.726598      0.726598    29.575879


### 3.4. Resumen del ensamble

Agregacion de metricas de validacion por fold y semilla, y guardado del estado del ensamble.

In [10]:
cols = ["eval_precision", "eval_recall", "eval_f1", "eval_accuracy", "eval_loss"]
agg = df_ensemble_results[cols].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "metric"})
print(agg.to_string(index=False))

summary = {c: {"mean": float(df_ensemble_results[c].mean()), "std": float(df_ensemble_results[c].std(ddof=0))} for c in cols}
summary["ensemble_size"] = len(ensemble_models)
with open(f"{RESULTS_DIR}/validation_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

        metric     mean      std      min      max
eval_precision 0.746085 0.020663 0.706496 0.777932
   eval_recall 0.775326 0.016377 0.747878 0.800097
       eval_f1 0.760344 0.016842 0.726598 0.781192
 eval_accuracy 0.962297 0.002797 0.959096 0.967114
     eval_loss 0.548082 0.073477 0.376966 0.633851


## 4. Inferencia

### 4.1. Funcion de inferencia por oraciones

Segmenta cada documento con spaCy, aplica el pipeline NER por oracion y reajusta los offsets al texto completo.

In [11]:
def sentence_based_ner(texto, pipeline_ner, nlp_spacy):
    """Inferencia NER por oraciones y ajuste de offsets al documento completo."""
    doc = nlp_spacy(texto)
    all_entities = []

    for sent in doc.sents:
        sent_text = sent.text
        sent_offset = sent.start_char

        entities = pipeline_ner(sent_text)

        for entity in entities:
            entity["start"] += sent_offset
            entity["end"] += sent_offset
            all_entities.append(entity)

    return all_entities

### 4.2. Ejecucion del ensamble

Lectura de los textos de test, inferencia con cada modelo del ensamble y agregacion de entidades por votacion mayoritaria.

In [12]:
ruta_txts = DATA_PATHS["text_files_test_dir"]
ruta_gs   = DATA_PATHS["gs_mentions_tsv"]

texts_by_filename = {
    f.replace(".txt", ""): open(os.path.join(ruta_txts, f), encoding="utf-8").read()
    for f in sorted(os.listdir(ruta_txts)) if f.endswith(".txt")
}

if not texts_by_filename:
    raise RuntimeError(f"No se encontraron archivos .txt en {ruta_txts}")
if not ensemble_models:
    raise RuntimeError("No hay modelos en el ensamble.")

vote_threshold = max(1, int(np.ceil(ENSEMBLE_VOTING_RATIO * len(ensemble_models))))
pred_file = f"{RESULTS_DIR}/predictions_ensemble_k{K_FOLDS}_s{len(SEEDS)}.tsv"
aggregated = defaultdict(int)
model_times = []
end_to_end_start = t0 = time.time()

print(f"Archivos test: {len(texts_by_filename)} | Modelos: {len(ensemble_models)} | Votos requeridos: {vote_threshold}")

for model_info in ensemble_models:
    fold, seed, model_dir = model_info["fold"], model_info["seed"], model_info["model_dir"]
    t_model = time.time()

    modelo_inf    = AutoModelForTokenClassification.from_pretrained(model_dir)
    tokenizer_inf = AutoTokenizer.from_pretrained(model_dir)
    nlp_ner = pipeline("ner", model=modelo_inf, tokenizer=tokenizer_inf, aggregation_strategy="simple")

    for filename, texto in texts_by_filename.items():
        for ent in sentence_based_ner(texto, nlp_ner, nlp_spacy):
            if ent["entity_group"] == "PROCEDIMIENTO":
                aggregated[(filename, int(ent["start"]), int(ent["end"]))] += 1

    elapsed = time.time() - t_model
    model_times.append(elapsed)
    print(f"  fold={fold} seed={seed}: {elapsed:.1f}s")

    del nlp_ner, tokenizer_inf, modelo_inf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Consenso por votacion
mark_counter = defaultdict(int)
final_rows = []
for (filename, off0, off1), votes in sorted(aggregated.items()):
    if votes < vote_threshold:
        continue
    mark_counter[filename] += 1
    final_rows.append({
        "filename": filename,
        "ann_id": f"T{mark_counter[filename]}",
        "label": "PROCEDIMIENTO",
        "start_span": off0, "end_span": off1,
        "text": texts_by_filename[filename][off0:off1],
    })

df_pred = pd.DataFrame(final_rows, columns=["filename", "ann_id", "label", "start_span", "end_span", "text"])
df_pred.to_csv(pred_file, sep="\t", index=False)

total_s = time.time() - t0
stats = {
    "archivos_procesados": len(texts_by_filename),
    "modelos_ensamblados": len(ensemble_models),
    "voting_ratio": ENSEMBLE_VOTING_RATIO,
    "votos_requeridos": vote_threshold,
    "entidades_candidatas": len(aggregated),
    "entidades_detectadas": len(df_pred),
    "inference_total_seconds": total_s,
    "inference_avg_file_seconds": total_s / len(texts_by_filename),
    "inference_avg_model_seconds": float(np.mean(model_times)),
}
with open(f"{RESULTS_DIR}/inference_stats_ensemble.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"Entidades detectadas: {len(df_pred)} | Tiempo total: {total_s:.1f}s | Predicciones: {pred_file}")

Archivos test: 250 | Modelos: 10 | Votos requeridos: 5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  fold=1 seed=123: 55.3s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

  fold=4 seed=4242: 52.5s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  fold=5 seed=123: 51.0s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  fold=5 seed=4242: 50.4s
Entidades detectadas: 3522 | Tiempo total: 527.6s | Predicciones: results_bsc-bio-ehr-es_kfold_multiseed/predictions_ensemble_k5_s2.tsv


## 5. Evaluacion

### 5.1. Evaluacion estricta por offsets

Comparacion de predicciones contra la referencia mediante coincidencia exacta de etiqueta y offsets de caracter.

In [13]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

df_gs   = pd.read_csv(ruta_gs,   sep="\t")
df_pred = pd.read_csv(pred_file, sep="\t")

set_gs   = set(zip(df_gs["filename"],   df_gs["label"],   df_gs["start_span"],   df_gs["end_span"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["start_span"], df_pred["end_span"]))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
precision, recall, fscore = prf(tp, fp, fn)
end_to_end_seconds = time.time() - end_to_end_start

strict_report = {
    "base_model": BASE_MODEL, "k_folds": K_FOLDS, "seeds": SEEDS,
    "ensemble_size": len(ensemble_models), "voting_ratio": ENSEMBLE_VOTING_RATIO,
    "tp": tp, "fp": fp, "fn": fn,
    "precision": precision, "recall": recall, "fscore": fscore,
    "end_to_end_seconds": end_to_end_seconds,
    "predictions_file": pred_file,
}
with open(f"{RESULTS_DIR}/strict_evaluation_ensemble.json", "w", encoding="utf-8") as f:
    json.dump(strict_report, f, ensure_ascii=False, indent=2)

print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {fscore:.4f}")
print(f"TP={tp} FP={fp} FN={fn} | End-to-end: {end_to_end_seconds:.1f}s")

Precision: 0.8157 | Recall: 0.7941 | F1: 0.8048
TP=2873 FP=649 FN=745 | End-to-end: 527.8s


### 5.2. Evaluacion por solapamiento (IoU)

Calculo de precision, recall y F1 bajo distintos umbrales de solapamiento entre spans predichos y de referencia.

In [14]:
EVAL_SUMMARY_JSON = f"{RESULTS_DIR}/overlap_eval_summary.json"
processed_files   = df_pred["filename"].unique()
df_gs_filt        = df_gs[df_gs["filename"].isin(processed_files)]

thresholds = [0.0, 0.5, 0.8]
results    = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}

for filename in processed_files:
    gs_ints   = list(zip(df_gs_filt[df_gs_filt["filename"] == filename]["start_span"],
                         df_gs_filt[df_gs_filt["filename"] == filename]["end_span"]))
    pred_ints = list(zip(df_pred[df_pred["filename"] == filename]["start_span"],
                         df_pred[df_pred["filename"] == filename]["end_span"]))

    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi,(p0,p1) in enumerate(pred_ints)
         for gi,(g0,g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )

    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi); matched_g.add(gi)
        tp = len(matched_p)
        results[t]["tp"] += tp
        results[t]["fp"] += len(pred_ints) - tp
        results[t]["fn"] += len(gs_ints)   - tp

report = {"Estricta": {**dict(zip(["tp","fp","fn"],[strict_report["tp"],strict_report["fp"],strict_report["fn"]])),
                       "precision": strict_report["precision"], "recall": strict_report["recall"], "fscore": strict_report["fscore"]}}
print(f"Estricta: P={strict_report['precision']:.4f} R={strict_report['recall']:.4f} F1={strict_report['fscore']:.4f}\n")

for t in thresholds:
    tp, fp, fn = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p, r, f1 = prf(tp, fp, fn)
    report[f"IoU >= {t}"] = {"tp": tp, "fp": fp, "fn": fn, "precision": round(p,4), "recall": round(r,4), "fscore": round(f1,4)}
    print(f"IoU >= {t}: P={p:.4f} R={r:.4f} F1={f1:.4f} | TP={tp} FP={fp} FN={fn}")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

Estricta: P=0.8157 R=0.7941 F1=0.8048

IoU >= 0.0: P=0.9225 R=0.8978 F1=0.9100 | TP=3249 FP=273 FN=370
IoU >= 0.5: P=0.8734 R=0.8500 F1=0.8615 | TP=3076 FP=446 FN=543
IoU >= 0.8: P=0.8242 R=0.8022 F1=0.8131 | TP=2903 FP=619 FN=716
